# Pertemuan 11 — Penalaran dalam Ketidakpastian: Probabilitas, Bayes, dan Bayesian Network
**PIF403 Pengantar Kecerdasan Buatan · Prodi Informatika Universitas Adzkia**

Aturan JIKA–MAKA pertemuan 10 menganggap dunia serba pasti. Kenyataannya gejala bisa samar, sensor bisa keliru, dan pengetahuan tidak pernah lengkap. Agen yang rasional tidak berpura-pura yakin: ia **mengukur keyakinannya dengan probabilitas** dan **memperbaruinya** setiap kali ada bukti baru.

**Setelah menyelesaikan notebook ini Anda mampu:**
1. menghitung probabilitas bersyarat dan menerapkan aturan Bayes;
2. menafsirkan pembaruan keyakinan (prior → posterior) dan menghindari kekeliruan *base rate*;
3. membangun Bayesian network kecil dan melakukan inferensi dengan enumerasi;
4. menyebutkan asumsi model dan batas penafsirannya.

Notebook ini adalah **tahap penalaran/evaluasi** menuju asesmen A6 (mini-project).

In [ ]:
NAMA = "tulis nama Anda"
NIM = "tulis NIM Anda"
print(f"Notebook P11 dikerjakan oleh: {NAMA} ({NIM})")

import itertools, random, math
from collections import Counter
import pandas as pd, matplotlib.pyplot as plt

## 1. Probabilitas dari data

Unggah berkas `sms_latihan.csv` dari LMS. Berkas ini berisi 170 pesan singkat berlabel `spam` atau `bukan_spam`.

> **Catatan kejujuran data:** pesan-pesan ini **data sintetis** yang disusun dosen dari pola kalimat berulang untuk keperluan latihan. Ini bukan data pesan asli, sehingga angka akurasi di notebook ini tidak mencerminkan kinerja di dunia nyata.

In [ ]:
import os

BERKAS = "sms_latihan.csv"
if not os.path.exists(BERKAS):
    from google.colab import files
    print("Unggah berkas:", BERKAS)
    files.upload()
assert os.path.exists(BERKAS), f"Berkas {BERKAS} belum diunggah. Jalankan ulang sel ini."

sms = pd.read_csv(BERKAS)
print(len(sms), "pesan")
print(sms["label"].value_counts().to_string())
sms.head()

In [ ]:
# Bagi data: 70% untuk menghitung probabilitas (latih), 30% disimpan untuk menguji
acak = sms.sample(frac=1, random_state=11).reset_index(drop=True)
batas = int(0.7 * len(acak))
latih, uji = acak.iloc[:batas], acak.iloc[batas:]

def punya(teks, kata):
    return kata in teks.split()

n = len(latih)
n_spam = (latih.label == "spam").sum()
p_spam = n_spam / n
print(f"P(spam)        = {n_spam}/{n} = {p_spam:.3f}   ← probabilitas PRIOR")

kata = "hadiah"
n_kata = latih.teks.apply(lambda t: punya(t, kata)).sum()
n_kata_spam = latih[latih.label == "spam"].teks.apply(lambda t: punya(t, kata)).sum()
print(f"P('{kata}')     = {n_kata}/{n} = {n_kata / n:.3f}")
print(f"P('{kata}' | spam) = {n_kata_spam}/{n_spam} = {n_kata_spam / n_spam:.3f}   ← probabilitas BERSYARAT")

### Probabilitas bersyarat
`P(A | B)` dibaca "probabilitas A **bila diketahui** B":

$$P(A \mid B) = \frac{P(A \wedge B)}{P(B)}$$

Kita sudah tahu `P('hadiah' | spam)`. Tetapi yang dibutuhkan penyaring spam adalah arah sebaliknya: `P(spam | 'hadiah')`. Kedua besaran itu **tidak sama**, dan menukarnya adalah kekeliruan yang sangat umum.

## 2. Aturan Bayes

$$P(H \mid E) = \frac{P(E \mid H)\,P(H)}{P(E)} \qquad\text{dengan}\qquad P(E) = P(E \mid H)P(H) + P(E \mid \neg H)P(\neg H)$$

| Istilah | Simbol | Arti |
|---|---|---|
| Prior | P(H) | keyakinan terhadap hipotesis **sebelum** melihat bukti |
| Likelihood | P(E \| H) | seberapa mungkin bukti muncul bila hipotesis benar |
| Posterior | P(H \| E) | keyakinan **sesudah** melihat bukti |

In [ ]:
def bayes(prior, p_bukti_jika_benar, p_bukti_jika_salah):
    """Mengembalikan posterior P(H | E)."""
    pembilang = p_bukti_jika_benar * prior
    penyebut = pembilang + p_bukti_jika_salah * (1 - prior)
    return pembilang / penyebut

def peluang_kata(kata, label):
    bagian = latih[latih.label == label]
    return bagian.teks.apply(lambda t: punya(t, kata)).mean()

for kata in ["hadiah", "pulsa", "menang", "link", "tugas"]:
    a, b = peluang_kata(kata, "spam"), peluang_kata(kata, "bukan_spam")
    print(f"{kata:9s} P(kata|spam) = {a:.2f}  P(kata|bukan) = {b:.2f}  →  P(spam|kata) = {bayes(p_spam, a, b):.2f}")

Kata yang hanya muncul pada pesan spam mendorong posterior ke 1. Kata yang juga muncul pada pesan biasa (misalnya `pulsa`, `menang`, `link`) memberi bukti yang lebih lemah. Kata khas pesan biasa seperti `tugas` justru menurunkan keyakinan.

> **Waspada terhadap angka 1,00 dan 0,00.** Posterior yang tepat satu atau nol di sini muncul karena datanya sedikit, bukan karena sistem benar-benar pasti. Kita perbaiki dengan *smoothing* pada bagian 3.

### Kekeliruan base rate
Kasus klasik: sebuah tes penyaring memiliki **sensitivitas 95%** (positif pada 95% orang yang sakit) dan **positif palsu 5%**. Penyakitnya diderita **1%** populasi. Seseorang dites positif. Berapa peluang ia benar-benar sakit?

Banyak orang menjawab 95%. Mari hitung.

In [ ]:
posterior = bayes(prior=0.01, p_bukti_jika_benar=0.95, p_bukti_jika_salah=0.05)
print(f"P(sakit | tes positif) = {posterior:.3f}")

# Cara frekuensi alami: bayangkan 10.000 orang
populasi = 10_000
sakit = populasi * 0.01; sehat = populasi - sakit
positif_benar = sakit * 0.95; positif_palsu = sehat * 0.05
print(f"\nDari 10.000 orang: {sakit:.0f} sakit dan {sehat:.0f} sehat")
print(f"Tes positif: {positif_benar:.0f} orang sakit + {positif_palsu:.0f} orang sehat = {positif_benar + positif_palsu:.0f}")
print(f"Jadi yang benar-benar sakit hanya {positif_benar:.0f} dari {positif_benar + positif_palsu:.0f} = {positif_benar / (positif_benar + positif_palsu):.1%}")

Hanya sekitar **16%**. Karena penyakitnya langka, orang sehat yang kebetulan positif jauh lebih banyak daripada orang sakit. Mengabaikan prior disebut **base rate fallacy**. Sistem AI yang memberi peringatan (deteksi penipuan, penyaringan pelamar, diagnosis) wajib melaporkan hal ini agar pengguna tidak menafsirkan "positif" sebagai "pasti".

### Pembaruan berurutan
Posterior hari ini menjadi prior esok hari. Bila tes diulang dan hasilnya positif lagi (dengan asumsi kedua tes saling bebas):

In [ ]:
keyakinan = [0.01]
for i in range(4):
    keyakinan.append(bayes(keyakinan[-1], 0.95, 0.05))
for i, k in enumerate(keyakinan):
    print(f"setelah {i} tes positif: P(sakit) = {k:.3f}")

plt.figure(figsize=(7, 3.6))
plt.plot(range(len(keyakinan)), keyakinan, "o-", color="#0E7C86")
plt.xticks(range(len(keyakinan))); plt.ylim(0, 1.05); plt.grid(alpha=0.3)
plt.xlabel("jumlah tes positif berturut-turut"); plt.ylabel("P(sakit)"); plt.title("Pembaruan keyakinan secara berurutan"); plt.show()

## 3. Naive Bayes: menggabungkan banyak bukti

Satu pesan memuat banyak kata. Untuk menggabungkan semuanya, Naive Bayes membuat **asumsi yang berani**: kemunculan setiap kata saling bebas bila kelasnya diketahui.

$$P(\text{kelas} \mid k_1, \dots, k_n) \;\propto\; P(\text{kelas}) \prod_i P(k_i \mid \text{kelas})$$

Asumsi ini jelas tidak sepenuhnya benar ("klik" dan "link" sering muncul bersama), tetapi sering cukup baik untuk klasifikasi. Dua rincian teknis:
- **Laplace smoothing**: tambahkan 1 pada setiap hitungan agar kata yang belum pernah terlihat tidak membuat peluang menjadi nol;
- **logaritma**: jumlahkan log-probabilitas agar hasil kali banyak angka kecil tidak menjadi nol di komputer.

In [ ]:
def latih_naive_bayes(data):
    hitung = {"spam": Counter(), "bukan_spam": Counter()}
    for _, r in data.iterrows():
        hitung[r.label].update(r.teks.split())
    kosakata = set(hitung["spam"]) | set(hitung["bukan_spam"])
    prior = (data.label.value_counts() / len(data)).to_dict()
    return {"hitung": hitung, "kosakata": kosakata, "prior": prior,
            "total": {k: sum(v.values()) for k, v in hitung.items()}}

def prediksi(model, teks, rinci=False):
    skor = {}
    for kelas in ("spam", "bukan_spam"):
        log_p = math.log(model["prior"][kelas])
        for kata in teks.split():
            if kata not in model["kosakata"]: continue                   # kata asing diabaikan
            p = (model["hitung"][kelas][kata] + 1) / (model["total"][kelas] + len(model["kosakata"]))
            log_p += math.log(p)
        skor[kelas] = log_p
    # ubah log-skor menjadi probabilitas posterior yang dinormalisasi
    m = max(skor.values()); eksp = {k: math.exp(v - m) for k, v in skor.items()}
    p_spam = eksp["spam"] / sum(eksp.values())
    return p_spam if rinci else ("spam" if p_spam >= 0.5 else "bukan_spam")

model = latih_naive_bayes(latih)
for teks in ["selamat anda menang hadiah mobil klik link berikut",
             "selamat ya kamu menang lomba pemrograman",
             "nanti transfer uang kas kelas ke rekening bendahara ya",
             "apa kabar"]:
    print(f"P(spam) = {prediksi(model, teks, rinci=True):.3f}  ← {teks}")

In [ ]:
uji = uji.assign(tebakan=uji.teks.apply(lambda t: prediksi(model, t)))
akurasi = (uji.tebakan == uji.label).mean()
print(f"Akurasi pada data uji: {akurasi:.1%} ({(uji.tebakan == uji.label).sum()} dari {len(uji)})")
print("\nMatriks kebingungan (baris = label sebenarnya, kolom = tebakan):")
print(pd.crosstab(uji.label, uji.tebakan).to_string())
dasar = max((uji.label == "spam").mean(), (uji.label == "bukan_spam").mean())
print(f"\nBaseline 'selalu tebak kelas terbanyak': {dasar:.1%}")

**Jangan terkecoh oleh akurasi yang tinggi.** Data ini sintetis dan berpola berulang, sehingga data uji sangat mirip data latih. Tiga hal yang harus selalu dilaporkan bersama angka akurasi:
1. **baseline** pembanding (di sini: selalu menebak kelas terbanyak);
2. **dari mana datanya** dan seberapa mirip dengan kondisi pemakaian sebenarnya;
3. **jenis kesalahannya**: pesan penting yang masuk folder spam jauh lebih merugikan daripada spam yang lolos.

Perhatikan pula pesan *"apa kabar"*: tidak ada satu kata pun yang dikenal model, sehingga jawabannya hanyalah prior. Model tidak tahu bahwa ia tidak tahu.

## 4. Bayesian network

Naive Bayes mengasumsikan semua bukti saling bebas. **Bayesian network** memberi kita cara menyatakan ketergantungan secara eksplisit:
- **simpul** = variabel acak;
- **panah** = pengaruh langsung (sering dibaca sebagai sebab → akibat);
- setiap simpul memiliki **tabel probabilitas bersyarat (CPT)** terhadap induknya.

Contoh: **mengapa saya terlambat kuliah?**

```
   Hujan      Kecelakaan
       \        /
        v      v
         Macet
           |
           v
       Terlambat
```

In [ ]:
# Setiap variabel: (daftar induk, CPT). CPT memetakan nilai induk -> P(variabel = True)
JARINGAN = {
    "Hujan":      ([], {(): 0.30}),
    "Kecelakaan": ([], {(): 0.10}),
    "Macet":      (["Hujan", "Kecelakaan"], {(True, True): 0.95, (True, False): 0.60,
                                             (False, True): 0.70, (False, False): 0.10}),
    "Terlambat":  (["Macet"], {(True,): 0.80, (False,): 0.10}),
}
URUTAN = list(JARINGAN)          # urutan topologis: induk selalu sebelum anak

def p_simpul(var, nilai, kejadian):
    induk, cpt = JARINGAN[var]
    p_benar = cpt[tuple(kejadian[i] for i in induk)]
    return p_benar if nilai else 1 - p_benar

def p_gabungan(kejadian):
    """Probabilitas satu kejadian lengkap = hasil kali P(simpul | induknya)."""
    hasil = 1.0
    for var in URUTAN:
        hasil *= p_simpul(var, kejadian[var], kejadian)
    return hasil

contoh = {"Hujan": True, "Kecelakaan": False, "Macet": True, "Terlambat": True}
print("P(hujan, tidak kecelakaan, macet, terlambat) =", round(p_gabungan(contoh), 4), "= 0.3 × 0.9 × 0.6 × 0.8")

semua = [dict(zip(URUTAN, k)) for k in itertools.product([True, False], repeat=len(URUTAN))]
print("Jumlah seluruh probabilitas gabungan =", round(sum(p_gabungan(k) for k in semua), 6))

Jaringan ini hanya butuh **8 angka** (1 + 1 + 4 + 2), padahal tabel gabungan penuh untuk empat variabel biner memerlukan 15 angka. Untuk 30 variabel selisihnya menjadi ratusan angka melawan lebih dari satu miliar.

### Inferensi dengan enumerasi
Untuk menjawab `P(X | bukti)`: jumlahkan probabilitas gabungan seluruh kejadian yang cocok dengan bukti, lalu normalisasi.

In [ ]:
def tanya(var, bukti=None):
    """Mengembalikan P(var = True | bukti) dengan menjumlahkan seluruh kejadian yang konsisten."""
    bukti = bukti or {}
    cocok = [k for k in semua if all(k[b] == v for b, v in bukti.items())]
    penyebut = sum(p_gabungan(k) for k in cocok)
    pembilang = sum(p_gabungan(k) for k in cocok if k[var])
    return pembilang / penyebut

print(f"P(Macet)                              = {tanya('Macet'):.3f}")
print(f"P(Terlambat)                          = {tanya('Terlambat'):.3f}")
print(f"P(Terlambat | Hujan)                  = {tanya('Terlambat', {'Hujan': True}):.3f}   ← penalaran sebab → akibat")
print(f"P(Hujan | Terlambat)                  = {tanya('Hujan', {'Terlambat': True}):.3f}   ← penalaran akibat → sebab (diagnostik)")
print(f"P(Kecelakaan | Terlambat)             = {tanya('Kecelakaan', {'Terlambat': True}):.3f}")
print(f"P(Kecelakaan | Terlambat, Hujan)      = {tanya('Kecelakaan', {'Terlambat': True, 'Hujan': True}):.3f}   ← explaining away")

Perhatikan dua baris terakhir. Setelah tahu kita terlambat, kecurigaan terhadap kecelakaan naik. Tetapi begitu tahu bahwa **hujan** turun, kecurigaan itu **turun lagi**: hujan sudah cukup menjelaskan kemacetan. Fenomena ini disebut **explaining away**, dan tidak dapat ditiru oleh aturan JIKA–MAKA biasa.

In [ ]:
skenario = {"tanpa bukti": {}, "terlambat": {"Terlambat": True}, "terlambat + hujan": {"Terlambat": True, "Hujan": True},
            "terlambat + tidak hujan": {"Terlambat": True, "Hujan": False}}
nilai = [tanya("Kecelakaan", b) for b in skenario.values()]
plt.figure(figsize=(8, 3.6))
plt.bar(list(skenario), nilai, color=["#8A9AA5", "#F2A024", "#0E7C86", "#D1495B"])
for i, v in enumerate(nilai): plt.text(i, v + 0.01, f"{v:.2f}", ha="center")
plt.ylabel("P(Kecelakaan | bukti)"); plt.title("Keyakinan berubah mengikuti bukti"); plt.ylim(0, max(nilai) + 0.1); plt.show()

In [ ]:
# Pemeriksaan dengan simulasi: bangkitkan banyak hari acak lalu hitung frekuensinya
rng = random.Random(0)
def hari_acak():
    k = {}
    for var in URUTAN:
        k[var] = rng.random() < p_simpul(var, True, k)
    return k
simulasi = [hari_acak() for _ in range(100_000)]
terlambat = [k for k in simulasi if k["Terlambat"]]
print(f"Simulasi : P(Hujan | Terlambat) ≈ {sum(k['Hujan'] for k in terlambat) / len(terlambat):.3f}")
print(f"Enumerasi: P(Hujan | Terlambat) = {tanya('Hujan', {'Terlambat': True}):.3f}")

## 5. Asumsi dan batas penafsiran
Setiap angka di notebook ini bergantung pada asumsi. Dokumentasi yang jujur menuliskannya.

| Yang diasumsikan | Risikonya bila keliru |
|---|---|
| Angka CPT (0,30; 0,60; ...) benar | hasil tampak presisi padahal masukannya tebakan |
| Struktur panah benar dan lengkap | variabel yang terlupa (misalnya jam berangkat) membuat kesimpulan bias |
| Panah berarti sebab-akibat | jaringan hanya menjamin hubungan probabilistik, bukan bukti kausal |
| Data latih mewakili kondisi pemakaian | model gagal ketika pola berubah |
| Bukti saling bebas (Naive Bayes, tes berulang) | keyakinan menjadi berlebihan |

---
## Latihan (tahap penalaran, formatif menuju A6)

### Latihan 1 — Hitung manual lalu periksa
Sebuah sistem deteksi plagiarisme menandai 90% tugas hasil salinan dan keliru menandai 8% tugas asli. Diperkirakan 5% tugas adalah salinan.
1. Hitung di lembar tugas: bila sebuah tugas ditandai, berapa peluang tugas itu benar-benar salinan?
2. Periksa jawaban Anda dengan fungsi `bayes`.
3. Apa konsekuensinya bila dosen langsung memberi nilai nol pada setiap tugas yang ditandai?

In [ ]:
# Jawaban Latihan 1

### Latihan 2 — Mengubah jaringan
1. Tambahkan simpul `BangunKesiangan` (prior 0,2) sebagai induk kedua dari `Terlambat`. Susun CPT baru untuk `Terlambat` yang masuk akal dan jelaskan alasan angkanya.
2. Hitung `P(Macet | Terlambat)` sebelum dan sesudah penambahan simpul. Mengapa nilainya berubah?

In [ ]:
# Jawaban Latihan 2

### Latihan 3 — Ketidakpastian pada domain Anda
Pada domain kelompok (pertemuan 9 dan 10), pilih **satu kesimpulan** yang sebenarnya tidak pasti.
1. Bangun Bayesian network kecil (3–5 simpul) untuk kesimpulan itu. Tuliskan sumber setiap angka CPT: data, literatur, atau perkiraan.
2. Jalankan minimal tiga kueri: satu prediktif, satu diagnostik, dan satu yang menunjukkan *explaining away* (bila strukturnya memungkinkan).
3. Bandingkan dengan keluaran aturan JIKA–MAKA pertemuan 10 untuk kasus yang sama. Kapan sistem Anda sebaiknya menjawab dengan probabilitas, dan kapan dengan aturan pasti?

In [ ]:
# Jawaban Latihan 3

### Refleksi
1. Mengapa P(bukti | hipotesis) tidak boleh dibaca sebagai P(hipotesis | bukti)? Beri satu contoh di luar notebook ini.
2. Angka CPT pada domain Anda sebagian besar adalah perkiraan. Bagaimana cara menyampaikan hal itu kepada pengguna tanpa membuat sistem tampak tidak berguna?
3. Sebutkan satu keputusan yang **tidak boleh** diotomatiskan penuh berdasarkan probabilitas dari sistem Anda. Mengapa manusia tetap harus memutuskan?

*Jawaban:*

---
**Pengumpulan:** `P11_NIM_Nama.ipynb` diunggah ke LMS. Notebook pertemuan 9, 10, dan 11 menjadi bahan mini-project pada pertemuan 12–16.